In [ ]:
!pip -q install -U "transformers>=5.1.0" "datasets" "accelerate" "peft" sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import re
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
import transformers, datasets, accelerate, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)

torch: 2.9.0+cu126
transformers: 5.1.0
datasets: 4.5.0
accelerate: 1.12.0
peft: 0.18.1


In [ ]:
df = pd.read_csv('/content/nudge_balanced_740.csv')
df = df[["source", "target"]].dropna()

df.sample(5)

,source,target
588,task: learning_nudge\ntone: encouraging\nstyle...,Show up for one focused lesson and keep it sim...
403,task: learning_nudge\ntone: encouraging\nstyle...,Show up for a 20-minute study session today. M...
692,task: learning_nudge\ntone: encouraging\nstyle...,Show up for a 20-minute study session now—then...
722,task: learning_nudge\ntone: encouraging\nstyle...,Spend one small project step right now. This e...
362,task: learning_nudge\ntone: encouraging\nstyle...,Focus on one focused session now—then stop. It...


In [ ]:
print("Rows:", len(df))
print("Empty sources:", (df["source"].astype(str).str.strip() == "").sum())
print("Empty targets:", (df["target"].astype(str).str.strip() == "").sum())

Rows: 740
Empty sources: 0
Empty targets: 0


In [ ]:
ds = Dataset.from_pandas(df).train_test_split(test_size=0.20, seed=42)
train_ds, val_ds = ds["train"], ds["test"]
print("Train:", len(train_ds), "Val:", len(val_ds))


Train: 592 Val: 148


In [ ]:
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.08,
    target_modules=["q","k","v","o"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


trainable params: 3,538,944 || all params: 251,116,800 || trainable%: 1.4093


In [ ]:
MAX_SOURCE_LEN = 160
MAX_TARGET_LEN = 64

def preprocess(batch):
    out = tokenizer(
        batch["source"],
        text_target=batch["target"],
        max_length=MAX_SOURCE_LEN,
        max_target_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    out["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in seq]
        for seq in out["labels"]
    ]
    return out

In [ ]:
train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)


Map:   0%|          | 0/592 [00:00<?, ? examples/s]

Map:   0%|          | 0/148 [00:00<?, ? examples/s]

In [ ]:
labels = train_tok[0]["labels"]
print("non -100 labels:", sum(t != -100 for t in labels), "out of", len(labels))
print("first 30 labels:", labels[:30])

non -100 labels: 28 out of 64
first 30 labels: [17578, 335, 676, 28, 5733, 18805, 7, 230, 318, 189, 35, 1190, 5, 148, 22, 162, 530, 490, 15290, 230, 318, 20779, 5689, 652, 3050, 49, 5, 1, -100, -100]


In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="/content/flan_nudge_balanced_lora",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    num_train_epochs=20,
    weight_decay=0.01,
    warmup_steps=80,
    predict_with_generate=True,
    eval_strategy="steps",
    eval_steps=150,
    save_strategy="steps",
    save_steps=150,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=25,
    max_grad_norm=1.0,
    fp16=False,
    report_to="none",
    load_best_model_at_end=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
150,1.173692,0.780486
300,0.585369,0.454566
450,0.495451,0.422725
600,0.459797,0.403365


TrainOutput(global_step=740, training_loss=0.9906773122581276, metrics={'train_runtime': 530.7222, 'train_samples_per_second': 22.309, 'train_steps_per_second': 1.394, 'total_flos': 2573827257139200.0, 'train_loss': 0.9906773122581276, 'epoch': 20.0})

In [ ]:
SAVE_DIR = "/content/flan_nudge_balanced_best"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

Saved to: /content/flan_nudge_balanced_best


In [ ]:
VERBS_RE = r"^(Start|Do|Take|Spend|Focus on|Show up for|Work on|Open|Finish|Keep)\b"

LOW_KEYWORDS_RE  = r"(easier later|later|over time|step by step|tomorrow|worth it later|next steps)"
HIGH_KEYWORDS_RE = r"(already|easier and easier|paying off|momentum|on the right track|keeps getting easier|compounding results)"
BAD_FOR_HIGH_RE  = r"(feels slow today|hard now|heavy today|tough now)"


In [ ]:
def passes_mood(text, mood="low"):
    words = len(text.split())
    if not (18 <= words <= 28):
        return False
    if not re.search(VERBS_RE, text):
        return False

    if str(mood).lower().startswith("high"):
        if re.search(BAD_FOR_HIGH_RE, text, re.I):
            return False
        if not re.search(HIGH_KEYWORDS_RE, text, re.I):
            return False
        return True
    else:
        if not re.search(LOW_KEYWORDS_RE, text, re.I):
            return False
        return True


In [ ]:
def build_prompt(idea, mood="low"):
    # Important: because training data now has "mood:" lines,
    # we keep the same format at inference.
    mood_line = "mood: high_motivation" if str(mood).lower().startswith("high") else "mood: low_motivation"

    return (
        "task: learning_nudge\n"
        "tone: encouraging\n"
        "style: micro_action_future_relief\n"
        f"{mood_line}\n"
        "constraints: start_with_verb; one_small_action; 18-28_words; no_quotes; no_emojis\n"
        f"idea: {idea}\n"
        "Write ONE notification."
    )


In [ ]:
def generate_nudge(idea, mood="low", n_candidates=8, max_rounds=3):
    prompt = build_prompt(idea, mood=mood)

    for _ in range(max_rounds):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=180).to(model.device)
        outs = model.generate(
            **inputs,
            max_new_tokens=45,
            do_sample=True,
            temperature=0.85,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            num_return_sequences=n_candidates,
        )
        candidates = [tokenizer.decode(o, skip_special_tokens=True).strip() for o in outs]
        good = [c for c in candidates if passes_mood(c, mood=mood)]
        if good:
            return good[0], candidates

    return candidates[0], candidates


In [ ]:
tests = [
    ("I feel tired and want to stop.", "low"),
    ("I feel good today and want to keep going.", "high"),
    ("I'm falling behind in my plan.", "low"),
    ("I'm motivated and want to push a bit harder today.", "high"),
]

for idea, mood in tests:
    best, cands = generate_nudge(idea, mood=mood)
    print("MOOD:", mood)
    print("IDEA:", idea)
    print("BEST:", best)
    print("-"*70)

MOOD: low
IDEA: I feel tired and want to stop.
BEST: Work on a 20-minute study session and keep it simple. Tough now, but you’re building momentum for tomorrow.
----------------------------------------------------------------------
MOOD: high
IDEA: I feel good today and want to keep going.
BEST: Focus on one small project step and keep it simple. You’re in motion now—stay consistent and ride the momentum.
----------------------------------------------------------------------
MOOD: low
IDEA: I'm falling behind in my plan.
BEST: Show up for a 20-minute study session now—then stop. It’s hard now, but this is what makes learning easier later.
----------------------------------------------------------------------
MOOD: high
IDEA: I'm motivated and want to push a bit harder today.
BEST: Work on one part of your roadmap and keep it simple. You’re on the right track, and it’s paying off.
----------------------------------------------------------------------


In [ ]:
!zip -r flan_nudge_balanced_best.zip flan_nudge_balanced_best


  adding: flan_nudge_balanced_best/ (stored 0%)
  adding: flan_nudge_balanced_best/adapter_config.json (deflated 58%)
  adding: flan_nudge_balanced_best/README.md (deflated 66%)
  adding: flan_nudge_balanced_best/tokenizer_config.json (deflated 83%)
  adding: flan_nudge_balanced_best/adapter_model.safetensors (deflated 7%)
  adding: flan_nudge_balanced_best/training_args.bin (deflated 53%)
  adding: flan_nudge_balanced_best/tokenizer.json (deflated 79%)


In [ ]:
!unzip flan_nudge_balanced_best.zip


Archive:  flan_nudge_balanced_best.zip
   creating: flan_nudge_balanced_best/
  inflating: flan_nudge_balanced_best/adapter_config.json  
  inflating: flan_nudge_balanced_best/README.md  
  inflating: flan_nudge_balanced_best/tokenizer_config.json  
  inflating: flan_nudge_balanced_best/adapter_model.safetensors  
  inflating: flan_nudge_balanced_best/training_args.bin  
  inflating: flan_nudge_balanced_best/tokenizer.json  


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE_MODEL = "google/flan-t5-base"
ADAPTER_PATH = "flan_nudge_balanced_best"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.08, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

In [ ]:
def build_prompt(idea, mood="low"):
    mood_line = "mood: high_motivation" if mood == "high" else "mood: low_motivation"

    return (
        "task: learning_nudge\n"
        "tone: encouraging\n"
        "style: micro_action_future_relief\n"
        f"{mood_line}\n"
        "constraints: start_with_verb; one_small_action; 18-28_words; no_quotes; no_emojis\n"
        f"idea: {idea}\n"
        "Write ONE notification."
    )

def generate_nudge(idea, mood="low"):
    prompt = build_prompt(idea, mood)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=True,
        temperature=0.85,
        top_p=0.9,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_nudge("I feel tired and want to stop.", mood="low"))
print(generate_nudge("I feel great today and want to push more.", mood="high"))


Do one focused lesson and keep it simple. It’s hard now, but this is what makes learning easier later.
Show up for a 20-minute study session right now. This is compounding—each step makes the next one easier.
